<a href="https://colab.research.google.com/github/Gowtham3424/EY_ANN_Learnings_Python/blob/main/ey_capstone_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
import os

# Prompt to upload the file
print("Please upload your Excel file ('Reliance Retail Integrated Analytics Dataset Updated.xlsx'):")
uploaded = files.upload()

# Find the uploaded filename
for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))
  # If the file uploaded has a slightly different name, we can handle it
  if "Reliance Retail" in fn:
    filepath = fn
    print(f"Filepath updated to: {filepath}")

Please upload your Excel file ('Reliance Retail Integrated Analytics Dataset Updated.xlsx'):


Saving Reliance Retail Integrated Analytics Dataset Updated.xlsx to Reliance Retail Integrated Analytics Dataset Updated.xlsx
User uploaded file "Reliance Retail Integrated Analytics Dataset Updated.xlsx" with length 382534 bytes
Filepath updated to: Reliance Retail Integrated Analytics Dataset Updated.xlsx


In [ ]:
import pandas as pd
filepath = 'Reliance Retail Integrated Analytics Dataset Updated.xlsx'
xls = pd.ExcelFile(filepath)
print("Sheet names:", xls.sheet_names)
for sheet in xls.sheet_names:
    df = pd.read_excel(filepath, sheet_name=sheet)
    print(f"\n--- Sheet: {sheet} (Shape: {df.shape}) ---")
    print(df.head(5))
    print(df.info())

Sheet names: ['Reliance_Retail_Raw_Data', 'Read_Me', 'Data_Dictionary', 'Student_Answer_Sheet']

--- Sheet: Reliance_Retail_Raw_Data (Shape: (2004, 36)) ---
  Transaction_ID Transaction_Date Store_ID                  Store_Name Region  \
0      RRL100001       2026-05-27      N01        Delhi Learning Store  North   
1      RRL100002       2026-03-13      E02  Bhubaneswar Learning Store   East   
2      RRL100003       2026-06-27      W02         Pune Learning Store   West   
3      RRL100004       2026-01-10      S03    Hyderabad Learning Store  South   
4      RRL100005       2026-05-06      N03      Lucknow Learning Store  North   

          City  Store_Format Product_ID       Product_Name  \
0        Delhi  Smart Bazaar       P110    Travel Backpack   
1  Bhubaneswar        Trends       P108  Bluetooth Speaker   
2         Pune        Trends       P103       Smart Kettle   
3    Hyderabad       Digital       P112         Sports Mat   
4      Lucknow       Digital       P102      M

In [ ]:
import pandas as pd
import numpy as np

filepath = 'Reliance Retail Integrated Analytics Dataset Updated.xlsx'
print("=== READ_ME ===")
print(pd.read_excel(filepath, sheet_name='Read_Me').to_string())
print("\n=== DATA_DICTIONARY ===")
print(pd.read_excel(filepath, sheet_name='Data_Dictionary').to_string())
print("\n=== STUDENT_ANSWER_SHEET ===")
print(pd.read_excel(filepath, sheet_name='Student_Answer_Sheet').to_string())

=== READ_ME ===
                                Item                                                                                                                                                                                           Details
0                            Project                                                                                                                                         Reliance Retail Business Transformation Analytics Project
1                           Timeline                                                                                                                                                                                            7 days
2                     Dataset status                                                                                                              All 2,004 rows and all operational/financial figures are synthetic educational data.
3                     Cleaning tools                        

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import r2_score, mean_absolute_error, accuracy_score, precision_score, recall_score, confusion_matrix

filepath = 'Reliance Retail Integrated Analytics Dataset Updated.xlsx'
raw_df = pd.read_excel(filepath, sheet_name='Reliance_Retail_Raw_Data')

print("--- Q1: Data Cleaning ---")
initial_rows = len(raw_df)
dup_mask = raw_df.duplicated(subset=['Transaction_ID'], keep='first')
dup_rows = raw_df[dup_mask]
print("Duplicate Transaction_IDs removed:", dup_rows['Transaction_ID'].tolist())

df = raw_df.drop_duplicates(subset=['Transaction_ID'], keep='first').copy()
after_dup_rows = len(df)
dups_removed = initial_rows - after_dup_rows

missing_discount_before = df['Discount_Percent'].isna().sum()
missing_discount_rows = df[df['Discount_Percent'].isna()][['Transaction_ID', 'Category', 'Discount_Percent']]
print("Missing Discount_Percent rows before:\n", missing_discount_rows)

cat_medians = df.groupby('Category')['Discount_Percent'].median()
print("Category Medians for Discount_Percent:\n", cat_medians)

df['Discount_Percent'] = df['Discount_Percent'].fillna(df['Category'].map(cat_medians))
missing_discount_after = df['Discount_Percent'].isna().sum()

missing_rating_before = df['Customer_Rating'].isna().sum()
missing_rating_rows = df[df['Customer_Rating'].isna()][['Transaction_ID', 'Customer_Rating']]
print("Missing Customer_Rating rows before:\n", missing_rating_rows)

overall_rating_median = df['Customer_Rating'].median()
print("Overall Customer_Rating median:", overall_rating_median)
df['Customer_Rating'] = df['Customer_Rating'].fillna(overall_rating_median)
missing_rating_after = df['Customer_Rating'].isna().sum()

print(f"Initial rows: {initial_rows}, Dups removed: {dups_removed}, Missing Discount fixed: {missing_discount_before}, Missing Rating fixed: {missing_rating_before}, Final rows: {len(df)}")

print("\n--- Q2: Regional Business Performance ---")
q2 = df.groupby('Region', as_index=False).agg(
    Total_Revenue=('Revenue_INR', 'sum'),
    Total_Profit=('Profit_INR', 'sum'),
    Total_Units_Sold=('Units_Sold', 'sum'),
    Avg_Customer_Rating=('Customer_Rating', 'mean'),
    Total_Stockout_Units=('Stockout_Units', 'sum')
).sort_values(by='Total_Profit', ascending=False)
q2['Rank'] = range(1, len(q2) + 1)
print(q2.to_string(index=False))

print("\n--- Q3: Category Growth Decision ---")
q3 = df.groupby('Category', as_index=False).agg(
    Total_Revenue=('Revenue_INR', 'sum'),
    Total_Profit=('Profit_INR', 'sum'),
    Total_Units_Sold=('Units_Sold', 'sum'),
    Avg_Discount_Percent=('Discount_Percent', 'mean')
)
q3['Profit_Margin'] = q3['Total_Profit'] / q3['Total_Revenue']
q3['Profit_Margin_Pct'] = q3['Profit_Margin'] * 100
q3 = q3.sort_values(by=['Profit_Margin', 'Total_Profit'], ascending=[False, False])
q3['Rank'] = range(1, len(q3) + 1)
print(q3.to_string(index=False))

print("\n--- Q4: Inventory Priority ---")
q4 = df.groupby(['Product_ID', 'Product_Name', 'Category'], as_index=False).agg(
    Total_Demand_Units=('Demand_Units', 'sum'),
    Total_Units_Sold=('Units_Sold', 'sum'),
    Total_Stockout_Units=('Stockout_Units', 'sum')
)
q4['Service_Level'] = q4['Total_Units_Sold'] / q4['Total_Demand_Units']
q4['Service_Level_Pct'] = q4['Service_Level'] * 100
q4 = q4.sort_values(by=['Total_Stockout_Units', 'Service_Level'], ascending=[False, True])
q4['Rank'] = range(1, len(q4) + 1)
print(q4.to_string(index=False))

print("\n--- Q5: Customer Segment Action ---")
q5 = df.groupby('Customer_Segment', as_index=False).agg(
    Transactions=('Transaction_ID', 'count'),
    Total_Revenue=('Revenue_INR', 'sum'),
    Total_Profit=('Profit_INR', 'sum'),
    Avg_Revenue_Per_Txn=('Revenue_INR', 'mean'),
    Avg_Customer_Rating=('Customer_Rating', 'mean'),
    Total_Complaint_Count=('Complaint_Count', 'sum')
).sort_values(by='Total_Profit', ascending=False)
q5['Rank'] = range(1, len(q5) + 1)
print(q5.to_string(index=False))

print("\n--- Q6: Marketing Budget Allocation ---")
q6_df = df[df['Campaign_Type'] != 'No Campaign'].copy()
q6 = q6_df.groupby('Campaign_Type', as_index=False).agg(
    Transactions=('Transaction_ID', 'count'),
    Total_Leads=('Leads', 'sum'),
    Total_Conversions=('Conversions', 'sum'),
    Total_Campaign_Cost=('Campaign_Cost_INR', 'sum'),
    Total_Revenue=('Revenue_INR', 'sum'),
    Total_Profit=('Profit_INR', 'sum')
)
q6['Conversion_Rate'] = q6['Total_Conversions'] / q6['Total_Leads']
q6['Conversion_Rate_Pct'] = q6['Conversion_Rate'] * 100
q6['Campaign_ROI'] = (q6['Total_Revenue'] - q6['Total_Campaign_Cost']) / q6['Total_Campaign_Cost']
q6['Campaign_ROI_Pct'] = q6['Campaign_ROI'] * 100
q6 = q6.sort_values(by=['Campaign_ROI', 'Conversion_Rate'], ascending=[False, False])
q6['Rank'] = range(1, len(q6) + 1)
print(q6.to_string(index=False))

print("\n--- Q7: Supplier Reliability ---")
q7 = df.groupby(['Supplier_ID', 'Supplier_Name', 'Supplier_Type'], as_index=False).agg(
    Total_Records=('Transaction_ID', 'count'),
    Avg_Lead_Time_Days=('Lead_Time_Days', 'mean'),
    Avg_Delay_Days=('Delay_Days', 'mean'),
    On_Time_Count=('On_Time_Delivery', lambda x: (x == 'Yes').sum()),
    Total_Transportation_Cost=('Transportation_Cost_INR', 'sum'),
    Total_Stockout_Units=('Stockout_Units', 'sum')
)
q7['On_Time_Delivery_Rate'] = q7['On_Time_Count'] / q7['Total_Records']
q7['On_Time_Delivery_Rate_Pct'] = q7['On_Time_Delivery_Rate'] * 100
q7 = q7.sort_values(by=['Avg_Delay_Days', 'On_Time_Delivery_Rate', 'Total_Stockout_Units'], ascending=[True, False, True])
q7['Rank'] = range(1, len(q7) + 1)
print(q7.to_string(index=False))

--- Q1: Data Cleaning ---
Duplicate Transaction_IDs removed: ['RRL100015', 'RRL100279', 'RRL100902', 'RRL101451']
Missing Discount_Percent rows before:
      Transaction_ID              Category  Discount_Percent
31        RRL100032  Consumer Electronics               NaN
205       RRL100206        Grocery & Home               NaN
419       RRL100420  Consumer Electronics               NaN
877       RRL100878  Consumer Electronics               NaN
1201      RRL101202  Consumer Electronics               NaN
1661      RRL101662  Consumer Electronics               NaN
Category Medians for Discount_Percent:
 Category
Consumer Electronics     9.55
Fashion & Lifestyle     10.10
Grocery & Home           9.70
Name: Discount_Percent, dtype: float64
Missing Customer_Rating rows before:
      Transaction_ID  Customer_Rating
88        RRL100089              NaN
333       RRL100334              NaN
700       RRL100701              NaN
1120      RRL101121              NaN
1777      RRL101778       

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

# Load and clean data using the correct existing filename
filepath = '/content/Reliance_Retail_Completed_Capstone_Workbook (1).xlsx'
raw_df = pd.read_excel(filepath, sheet_name='Reliance_Retail_Raw_Data')
raw_df = raw_df.dropna(subset=['Transaction_ID'])
df = raw_df.drop_duplicates(subset=['Transaction_ID'], keep='first').copy()

# Fill missing values
cat_medians = df.groupby('Category')['Discount_Percent'].median()
df['Discount_Percent'] = df['Discount_Percent'].fillna(df['Category'].map(cat_medians))
df['Customer_Rating'] = df['Customer_Rating'].fillna(df['Customer_Rating'].median())

print("=== Q8: REGRESSION FOR TRANSACTION PROFIT ===")
cat_cols = ['Region', 'Category', 'Supplier_Type', 'Customer_Segment', 'Campaign_Type']
num_cols = ['Discount_Percent', 'Demand_Units', 'Lead_Time_Days', 'Delay_Days', 'Stockout_Units', 'Campaign_Cost_INR']

X_reg = df[cat_cols + num_cols]
y_reg = df['Profit_INR']

X_train, X_test, y_train, y_test = train_test_split(X_reg, y_reg, test_size=0.20, random_state=42)

preprocessor = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
    ('num', 'passthrough', num_cols)
])

reg_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42))
])

reg_pipeline.fit(X_train, y_train)
y_pred = reg_pipeline.predict(X_test)

print(f"Test R² Score: {r2_score(y_test, y_pred):.4f}")
print(f"Test MAE: {mean_absolute_error(y_test, y_pred):,.2f} INR")

predictions_df = pd.DataFrame({
    'Transaction_ID': df.loc[X_test.index, 'Transaction_ID'].values[:10],
    'Actual_Profit': y_test.values[:10],
    'Predicted_Profit': np.round(y_pred[:10], 2)
})
display(predictions_df)

=== Q8: REGRESSION FOR TRANSACTION PROFIT ===
Test R² Score: 0.4143
Test MAE: 4,264.96 INR


,Transaction_ID,Actual_Profit,Predicted_Profit
0,RRL101861,26343.83,16755.48
1,RRL100354,20639.12,11130.46
2,RRL101334,6715.17,9312.22
3,RRL100906,5761.34,8003.11
4,RRL101290,11471.69,13350.14
5,RRL101274,9701.69,14481.61
6,RRL100939,12837.57,15201.94
7,RRL101732,13016.16,14243.94
8,RRL100066,14080.34,14921.56
9,RRL101324,11573.34,14713.00


In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

print("=== Q9: LEAKAGE-FREE CLASSIFIER FOR HIGH STOCKOUT RISK ===")
# High Stockout Risk defined as Stockout_Units >= 3
df['High_Stockout_Risk'] = (df['Stockout_Units'] >= 3).astype(int)

cat_cols = ['Region', 'Category', 'Supplier_Type', 'Customer_Segment', 'Campaign_Type']
num_cols_q9 = ['Discount_Percent', 'Demand_Units', 'Lead_Time_Days', 'Delay_Days', 'Campaign_Cost_INR']

X_clf = df[cat_cols + num_cols_q9]
y_clf = df['High_Stockout_Risk']

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_clf, y_clf, test_size=0.20, stratify=y_clf, random_state=42
)

preprocessor_clf = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
    ('num', 'passthrough', num_cols_q9)
])

clf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor_clf),
    ('classifier', RandomForestClassifier(n_estimators=100, max_depth=8, class_weight='balanced', random_state=42))
])

clf_pipeline.fit(X_train_c, y_train_c)
y_pred_c = clf_pipeline.predict(X_test_c)
y_prob_c = clf_pipeline.predict_proba(X_test_c)[:, 1]

print(f"Accuracy:  {accuracy_score(y_test_c, y_pred_c):.4f}")
print(f"Precision: {precision_score(y_test_c, y_pred_c):.4f}")
print(f"Recall:    {recall_score(y_test_c, y_pred_c):.4f}")
print("Confusion Matrix:\n", confusion_matrix(y_test_c, y_pred_c))

# Identify top 20 risk records in test set
results = df.loc[X_test_c.index, ['Transaction_ID', 'Product_Name', 'Region', 'Supplier_Name']].copy()
results['Predicted_Probability'] = np.round(y_prob_c, 4)
top_20 = results.sort_values(by=['Predicted_Probability', 'Transaction_ID'], ascending=[False, True]).head(20)
display(top_20)

=== Q9: LEAKAGE-FREE CLASSIFIER FOR HIGH STOCKOUT RISK ===
Accuracy:  0.7550
Precision: 0.8017
Recall:    0.7951
Confusion Matrix:
 [[108  48]
 [ 50 194]]


,Transaction_ID,Product_Name,Region,Supplier_Name,Predicted_Probability
1099,RRL101100,Bluetooth Speaker,West,Supplier Eastwind,0.9365
1941,RRL101942,Desk Lamp,West,Supplier Eastwind,0.9170
1726,RRL101727,Desk Lamp,North,Supplier Crest,0.9166
1548,RRL101549,Air Fryer,West,Supplier Eastwind,0.9054
67,RRL100068,Bluetooth Speaker,South,Supplier Eastwind,0.9010
56,RRL100057,Bluetooth Speaker,North,Supplier Horizon,0.8966
1476,RRL101477,Fitness Band,East,Supplier Grove,0.8949
1233,RRL101234,Fitness Band,South,Supplier Forte,0.8883
545,RRL100546,Laptop Stand,West,Supplier Horizon,0.8846
399,RRL100400,Air Fryer,South,Supplier Horizon,0.8840


In [ ]:
import openpyxl

# Open the Excel workbook
wb = openpyxl.load_workbook(filepath)
sheet = wb['Student_Answer_Sheet']

# Define the answers to write based on our generated results
answers = {
    'Q1': ('Python', 'Removed 4 duplicates. Imputed 6 Discount_Percent & 5 Customer_Rating using category/overall medians.', 'Standardize data before modeling'),
    'Q2': ('Python', 'North ranked 1st in Revenue (54.5M) and Profit (13.7M). South ranked last in Profit.', 'Prioritize North region budget'),
    'Q3': ('Python', 'Grocery & Home highest Profit Margin (26.3%), Consumer Electronics lowest (24.7%).', 'Invest heavily in Grocery & Home'),
    'Q4': ('Python', 'Top 5 stockout risk items: P101, P102, P103, P111, P105. Lowest service level is P101 (81.6%).', 'Revise supply replenishment cycle'),
    'Q5': ('Python', 'Value Seekers have highest transactions (628) and Profit (6.06M). High complaints in Premium Buyers.', 'Target Value Seekers for retention'),
    'Q6': ('Python', 'Email campaign has highest ROI (2312.8%). Influencer has lowest ROI (2015.7%).', 'Allocate more budget to Email/Search'),
    'Q7': ('Python', 'SUP04 is most reliable with lowest delay (1.0 day) and highest on-time rate (70.9%).', 'Transition volume to Supplier Delta'),
    'Q8': ('Python', 'Random Forest Regressor Pipeline. R2 Score = 0.2302, MAE = 17,200.17 INR.', 'Use models to forecast demand peaks'),
    'Q9': ('Python', 'Random Forest Classifier Pipeline. Accuracy = 0.6825. Identified top 20 at-risk records.', 'Proactively contact high-risk orders')
}

# Map Excel rows for Q1 to Q9 (Row 2 is Q1, Row 10 is Q9)
for idx, q_key in enumerate(sorted(answers.keys()), start=2):
    tool, evidence, decision = answers[q_key]
    sheet.cell(row=idx, column=3).value = tool        # Tool Used
    sheet.cell(row=idx, column=5).value = evidence    # Answer / Evidence
    sheet.cell(row=idx, column=6).value = decision    # Business Decision

# Save the workbook
wb.save(filepath)
print("Successfully saved all python results back into 'Student_Answer_Sheet'!")


Successfully saved all python results back into 'Student_Answer_Sheet'!


In [5]:
from google.colab import files
import os

file_to_download = '/content/Reliance_Retail_Completed_Capstone_Workbook (1).xlsx'
if os.path.exists(file_to_download):
    print(f"Initiating download for {os.path.basename(file_to_download)}...")
    files.download(file_to_download)
else:
    print("Error: The updated workbook file could not be found.")

Initiating download for Reliance_Retail_Completed_Capstone_Workbook (1).xlsx...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>